# UC Food Waste 데이터 확인

원본 파일은 수정하지 않고 RGB/Mask 짝, 분할 수, 샘플 배열을 읽기 전용으로 확인합니다.

Colab에서는 Drive를 마운트한 뒤 `UC_FOODWASTE_DATA_ROOT`와 `UC_FOODWASTE_NPZ_DIR`를 설정하세요.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
from PIL import Image
PROJECT_ROOT = Path(os.environ.get('UC_FOODWASTE_REPO_ROOT', Path.cwd()))
DATA_ROOT = Path(os.environ.get('UC_FOODWASTE_DATA_ROOT', PROJECT_ROOT / 'data'))
NPZ_DIR = Path(os.environ.get('UC_FOODWASTE_NPZ_DIR', DATA_ROOT / 'data_processed_multimodal' / 'images'))
RGB_DIR = DATA_ROOT / 'data_processed' / 'images'
MASK_DIR = DATA_ROOT / 'data_processed' / 'masks'
SPLIT_CSV = Path(os.environ.get('UC_FOODWASTE_SPLIT_CSV', PROJECT_ROOT / 'metadata' / 'data_split.csv'))
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('NPZ_DIR:', NPZ_DIR)

In [ ]:
assert SPLIT_CSV.exists(), f'Split CSV not found: {SPLIT_CSV}'
df = pd.read_csv(SPLIT_CSV)
required = {'filename', 'split', 'group', 'block_id'}
missing_columns = required - set(df.columns)
assert not missing_columns, f'Missing CSV columns: {missing_columns}'
print('Rows:', len(df))
print('Split counts:\n', df['split'].value_counts().sort_index())
print('Group counts:\n', df['group'].value_counts().sort_index())
assert len(df) == 542
assert df['filename'].nunique() == 542
assert df['split'].value_counts().to_dict() == {'train': 374, 'val': 84, 'test': 84}

In [ ]:
def stems(folder, suffix):
    return {p.stem for p in folder.glob(f'*{suffix}')} if folder.exists() else set()
csv_stems = set(df['filename'].astype(str))
rgb_stems = stems(RGB_DIR, '.jpg')
mask_stems = stems(MASK_DIR, '.png')
print(f'RGB JPGs: {len(rgb_stems)} | Masks: {len(mask_stems)}')
if RGB_DIR.exists() and MASK_DIR.exists():
    print('RGB missing:', sorted(csv_stems - rgb_stems)[:10])
    print('Mask missing:', sorted(csv_stems - mask_stems)[:10])
    print('RGB extra:', sorted(rgb_stems - csv_stems)[:10])
    print('Mask extra:', sorted(mask_stems - csv_stems)[:10])
    assert rgb_stems == csv_stems
    assert mask_stems == csv_stems
else:
    print('RGB or Mask folder is not connected; set UC_FOODWASTE_DATA_ROOT.')
if mask_stems:
    sample_mask = MASK_DIR / f'{sorted(mask_stems)[0]}.png'
    with Image.open(sample_mask) as m:
        arr = np.asarray(m.convert('L'))
    print('Sample mask:', sample_mask.name, 'shape=', arr.shape, 'values=', np.unique(arr)[:20])

In [ ]:
npz_files = sorted(NPZ_DIR.glob('*.npz')) if NPZ_DIR.exists() else []
print('NPZ files:', len(npz_files))
if npz_files:
    sample = npz_files[0]
    with np.load(sample) as z:
        print('Sample:', sample.name, 'keys=', z.files)
        arr = z['image']
    print('Array:', arr.shape, arr.dtype, 'finite=', bool(np.isfinite(arr).all()))
    print('Per-channel min/max (first 5):')
    print(np.stack([arr.min((0,1)), arr.max((0,1))], axis=1)[:5])
    assert arr.shape == (512, 768, 19)
    assert len(npz_files) == 542
else:
    print('NPZ folder is not connected. Set UC_FOODWASTE_NPZ_DIR to the shared Drive images folder.')